In [1]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision import datasets, transforms, models
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ==== Setup ====
data_dir = r"C:\Users\sahba\Downloads\archive (2)\IMG_CLASSES"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==== Transform ====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
class_names = full_dataset.classes
num_classes = len(class_names)

# ==== Class Weights ====
class_counts = [2697, 2215, 6946, 1827, 1644, 1888]
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = class_weights.to(device)

# ==== Cross Validation Setup ====
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
all_fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(full_dataset)):
    print(f"\n\n==== Fold {fold + 1}/5 ====")

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    # Load pretrained ResNet-50
    model = models.resnet50(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer3.parameters():
        param.requires_grad = True
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Replace classifier head
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(512, num_classes)
    )
    model = model.to(device)

    # Loss, Optimizer, Scheduler
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = optim.Adam([
        {'params': model.layer3.parameters(), 'lr': 1e-6},
        {'params': model.layer4.parameters(), 'lr': 1e-5},
        {'params': model.fc.parameters(), 'lr': 1e-3}
    ])
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

    # Training Loop
    for epoch in range(20):
        print(f"\nEpoch {epoch + 1}")
        model.train()
        total, correct, running_loss = 0, 0, 0.0

        for images, labels in tqdm(train_loader, leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            running_loss += loss.item()

        print(f"Train Acc: {100 * correct / total:.2f}%, Loss: {running_loss / len(train_loader):.4f}")

        # Validation
        model.eval()
        total, correct, val_loss = 0, 0, 0.0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                val_loss += loss.item()
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_acc = 100 * correct / total
        print(f"Val Acc: {val_acc:.2f}%, Loss: {val_loss / len(val_loader):.4f}")
        scheduler.step(val_loss)

    # Store fold metrics
    report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
    all_fold_metrics.append(report)

# ==== Summary ====
avg_acc = np.mean([m['accuracy'] for m in all_fold_metrics])
print(f"\n✅ Average Accuracy over 5 folds: {avg_acc * 100:.2f}%")


Using device: cuda


==== Fold 1/5 ====


c:\Users\sahba\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\sahba\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
c:\Users\sahba\AppData\Local\Programs\Python\Python39\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Epoch 1


Train Acc: 70.62%, Loss: 1.1046
Val Acc: 82.18%, Loss: 1.0490

Epoch 2


Train Acc: 81.03%, Loss: 0.9438
Val Acc: 85.64%, Loss: 0.9854

Epoch 3


...

Train Acc: 95.86%, Loss: 0.6300
Val Acc: 92.06%, Loss: 0.8906

Epoch 17


Train Acc: 96.50%, Loss: 0.6193
Val Acc: 92.18%, Loss: 0.8799

Epoch 18


Train Acc: 96.33%, Loss: 0.6186
Val Acc: 92.47%, Loss: 0.8700

Epoch 19


Train Acc: 96.45%, Loss: 0.6160
Val Acc: 91.77%, Loss: 0.8972

Epoch 20


Train Acc: 96.91%, Loss: 0.6045
Val Acc: 92.47%, Loss: 0.8692

✅ Average Accuracy over 5 folds: 92.47%
